# Stage 1 setup and Drive check

Mount Google Drive, install the lightweight CPU dependencies, and verify the post-query Text-to-Visualization package import.

In [ ]:
from __future__ import annotations

import json
import os
import platform
from datetime import datetime, timezone
from pathlib import Path

from google.colab import drive

mount_error = None
try:
    drive.mount('/content/drive', force_remount=False)
except Exception as exc:
    mount_error = repr(exc)
    if not Path('/content/drive/MyDrive').exists():
        print('STAGE1_DRIVE_SETUP_BLOCKED')
        print(json.dumps({
            'status': 'blocked',
            'reason': 'drive.mount failed and /content/drive/MyDrive is not available',
            'mount_error': mount_error,
        }, ensure_ascii=False, indent=2))
        raise
    print('[warn] drive.mount failed, but /content/drive/MyDrive already exists; continuing with existing mount')

drive_root = Path(os.environ.get('T2V_DRIVE_ROOT', '/content/drive/MyDrive/diploma/petr_text_to_visualization_part'))
deprecated_non_diploma_root = Path('/content/drive/MyDrive/petr_text_to_visualization_part')
drive_root.mkdir(parents=True, exist_ok=True)

print('STAGE1_DRIVE_SETUP_OK')
print(json.dumps({
    'drive_root': str(drive_root),
    'drive_root_exists': drive_root.exists(),
    'deprecated_non_diploma_root': str(deprecated_non_diploma_root),
    'deprecated_non_diploma_root_exists': deprecated_non_diploma_root.exists(),
    'deprecated_non_diploma_root_note': 'not used for artifacts',
    'mount_error': mount_error,
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'python': platform.python_version(),
}, ensure_ascii=False, indent=2))


In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

repo_root = Path(os.environ.get('T2V_REPO_ROOT', Path.cwd())).resolve()
requirements = repo_root / 'requirements.txt'

if not requirements.exists():
    print(f'[warn] requirements.txt not found at {requirements}; set T2V_REPO_ROOT to the repository root before installing')
elif os.environ.get('T2V_SKIP_PIP_INSTALL', '0') == '1':
    print('[skip] T2V_SKIP_PIP_INSTALL=1')
else:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(repo_root)])
    print('STAGE1_DEPENDENCIES_OK')


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import t2v_eval
from t2v_eval.data.schema import FieldMetadata, T2VExample

drive_root = Path(os.environ.get('T2V_DRIVE_ROOT', '/content/drive/MyDrive/diploma/petr_text_to_visualization_part'))
for folder in ['datasets/raw', 'datasets/processed', 'datasets/samples', 'runs', 'report_materials']:
    (drive_root / folder).mkdir(parents=True, exist_ok=True)

example = T2VExample(
    example_id='stage1_smoke',
    benchmark='smoke',
    query='Show total sales by month',
    table_path=str(drive_root / 'datasets/samples/stage1_smoke.csv'),
    metadata={'fields': [FieldMetadata(name='sales', dtype='number', role='measure').to_dict()]},
    gold_spec={'mark': 'bar'},
)

print('STAGE1_IMPORT_AND_PATHS_OK')
print(json.dumps({
    'package_version': t2v_eval.__version__,
    'drive_root': str(drive_root),
    'example': example.to_dict(),
}, ensure_ascii=False, indent=2))
